# Robusta–chicory regression

Chicory content in robusta coffee, from `training_samples.csv`. Arabica (AB100) is dropped: it is a
different species, the model has no arabica mixtures to learn from, and including a single pure arabica
plate would only distort the fit. What it revealed is recorded in section 9 as a limitation.

## What this version does differently

Three changes, each tested on your data rather than assumed:

**1. Patches are aggregated into super-patches before modelling.** Your 12×12 pixel patch spans about
8.8 mm, and at 0.5–0.7 mm per pixel with 0.3–1.0 mm particles that is still small enough for particle
lottery to dominate each sample. Averaging groups of patches cuts the error roughly in half:

| unit | RMSECV (unit) | RMSECV (plate) |
|---|---|---|
| 1 patch | 5.21 | 3.24 |
| 4 patches | 3.18 | 2.20 |
| 8 patches | 2.68 | 2.07 |
| 16 patches | 2.79 | 2.62 |

The gain plateaus at 8 and reverses at 16, where there are too few independent units left. Grouping here
is **spatial** — patches are clustered by their position in the plate, so a super-patch is a contiguous
area rather than a random scatter. That makes it deterministic and physically meaningful.

**2. Outliers are trimmed within each plate**, worst 10 % by Q residual. Trimming globally instead would
be a serious mistake: day-1 spectra carry systematically higher Q, so a global filter **deletes every
RB100 patch** and silently removes the only 0 % level. Ranking inside each plate avoids that.

**3. Results are reported per plate.** If the deliverable is "this sample contains X % chicory", the plate
is the unit of interest, and its error is already much lower than the per-patch figure.

## Preprocessing: absorbance + offset removal

The two sessions differ by a uniform brightness factor of 1.147 — multiplicative, from packing density or
sample height. In absorbance, `A = log10(1/R)`, a multiplicative factor becomes an additive offset, and
subtracting each spectrum's own mean removes it exactly.

Verified: scaling a spectrum's reflectance anywhere from ×0.85 to ×1.15 changes the prediction by **zero**.
It also drops the session bias from +10.5 pp (SNV) to −1.6 pp.

Alternatives were tested and rejected. Derivatives push the session bias to +20 pp. VIP-based wavelength
selection improves within-day cross-validation but pushes the bias to +6.6 — and every sample you want to
predict comes from the other session, so transfer matters more than the CV score.

## 1. Load

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import f as f_dist, chi2
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "figure.facecolor": "white",
                     "axes.grid": True, "grid.alpha": .3})

# ------------------------------- EDIT -------------------------------
CSV   = Path("./training_samples.csv")
OUT   = Path("./rb_ch_model")
WL_MIN, WL_MAX = 1000.0, 2450.0
TRIM_WORST_PCT   = 10     # drop this % of patches per plate by Q residual
# Super-patch size and latent variables are chosen automatically in section 3.
# Set these to a number to override.
PATCHES_PER_UNIT = "auto"
N_COMP           = "auto"
N_PERM  = 300
# ---------------------------------------------------------------------

OUT.mkdir(exist_ok=True, parents=True)
RNG = np.random.default_rng(0)

raw = pd.read_csv(CSV)
raw = raw[raw.label != "AB100"].reset_index(drop=True)      # arabica out

wl_all = np.array([float(c) for c in raw.columns if c.replace(".", "", 1).isdigit()])
cols_all = [c for c in raw.columns if c.replace(".", "", 1).isdigit()]
keep = (wl_all >= WL_MIN) & (wl_all <= WL_MAX)
WL = wl_all[keep]
cols = [c for c, k in zip(cols_all, keep) if k]

R = np.clip(raw[cols].values.astype(float), 1e-3, None)
meta = raw[[c for c in raw.columns if c not in cols_all]].copy()

print(f"{len(raw)} patches after dropping arabica | {len(WL)} bands "
      f"({WL[0]:.0f}-{WL[-1]:.0f} nm, {int((~keep).sum())} dropped)")
print()
print(meta.groupby(["role", "label"], dropna=False)
      .agg(patches=("sample_id", "size"), plates=("group", "nunique"),
           chicory=("chicory_pct", "first")).to_string())

In [ ]:
# absorbance + per-spectrum offset removal
Z = np.log10(1.0 / R)
Z = Z - Z.mean(1, keepdims=True)

cal = (meta.role == "calibration").values
y_all = meta.chicory_pct.values

fig, axs = plt.subplots(1, 2, figsize=(13, 3.6))
cmap = plt.get_cmap("viridis")
for lv in sorted(meta.loc[cal, "chicory_pct"].unique()):
    s = cal & (y_all == lv)
    axs[0].plot(WL, R[s].mean(0), lw=1.1, color=cmap(lv / 60), label=f"{lv:.0f}%")
    axs[1].plot(WL, Z[s].mean(0), lw=1.1, color=cmap(lv / 60))
axs[0].set_title("raw reflectance — the 0 % level sits apart (different session)")
axs[1].set_title("absorbance + offset removal — the session gap is gone")
axs[0].legend(fontsize=6, ncol=2, title="chicory")
for ax in axs: ax.set_xlabel("wavelength (nm)")
plt.tight_layout(); plt.savefig(OUT / "01_preprocessing.png", dpi=130, bbox_inches="tight"); plt.show()

## 2. Trim, then aggregate into super-patches

Order matters: the Q residual needs a model to be measured against, so a preliminary PLS is fitted on all
calibration patches first, used only to rank patches within their own plate.

Aggregation clusters each plate's surviving patches by position with k-means, so a super-patch covers a
contiguous piece of the powder surface rather than a random scatter across it. Deterministic, and it
behaves like a genuinely larger patch.

In [ ]:
prelim = PLSRegression(n_components=4).fit(Z[cal], y_all[cal])   # only used to rank patches
mu0 = Z[cal].mean(0)
T0 = (Z - mu0) @ prelim.x_rotations_
Qres = np.sum(((Z - mu0) - T0 @ prelim.x_loadings_.T) ** 2, axis=1)
meta["Q"] = Qres

# rank WITHIN each plate -- a global cut would delete day-1 samples wholesale
rank = meta.groupby("group")["Q"].rank(pct=True).values
keep_patch = rank <= (1 - TRIM_WORST_PCT / 100)
print(f"kept {keep_patch.sum()} of {len(meta)} patches "
      f"({100*keep_patch.mean():.0f} %) after trimming the worst {TRIM_WORST_PCT} % per plate")
print("patches removed per plate:")
print(meta.loc[~keep_patch].groupby("label").size().rename("dropped").to_string())


def aggregate(Zin, meta_in, mask, k, seed=0):
    # Spatially contiguous super-patches: k-means on patch position within each plate.
    if k <= 1:
        idx = [np.array([i]) for i in np.where(mask)[0]]
    else:
        idx = []
        for g in pd.unique(meta_in.group.values):
            ii = np.where((meta_in.group.values == g) & mask)[0]
            if len(ii) < k: continue
            ngroups = max(1, len(ii) // k)
            xy = meta_in.loc[ii, ["patch_row", "patch_col"]].values.astype(float)
            lab = KMeans(n_clusters=ngroups, n_init=5, random_state=seed).fit_predict(xy)
            idx += [ii[lab == c] for c in range(ngroups) if (lab == c).sum() > 0]
    Zs = np.array([Zin[i].mean(0) for i in idx])
    first = np.array([i[0] for i in idx])
    info = meta_in.iloc[first].reset_index(drop=True).copy()
    info["n_patches"] = [len(i) for i in idx]
    return Zs, info

## 3. Choosing the super-patch size and the number of latent variables

Cross-validation is **leave-one-concentration-out**: all units of one level are held out, the model fitted
on the rest, the held-out level predicted. It is the only split where the held-out concentration is truly
absent from training. A random split would place neighbouring super-patches from the same plate on both
sides and report something near perfect.

Two scores are tracked, and they answer different questions:

- **unit RMSECV** — error on one super-patch. Depends on the aggregation size, so it is not comparable
  across rows on its own.
- **plate RMSECV** — unit predictions averaged per plate first. The same quantity at every aggregation
  size, so this is the fair comparison, and it is the number that matters if you report per sample.

In [ ]:
CAL_LABELS = ["RB100", "RB[90:10]", "RB[80:20]", "RB[70:30]", "RB[60:40]", "RB[50:50]"]

def score(Zs, info, nc):
    c = info.label.isin(CAL_LABELS).values
    Zc, yc, gc = Zs[c], info.chicory_pct.values[c], info.group.values[c]
    p = np.full(len(yc), np.nan)
    for tr, te in LeaveOneGroupOut().split(Zc, yc, groups=yc):
        p[te] = PLSRegression(n_components=nc).fit(Zc[tr], yc[tr]).predict(Zc[te]).ravel()
    pl = pd.DataFrame(dict(g=gc, y=yc, p=p)).groupby(["g", "y"], as_index=False)["p"].mean()
    d2 = c & (info.day.values == 2)
    m2 = PLSRegression(n_components=nc).fit(Zs[d2], info.chicory_pct.values[d2])
    bias = float(m2.predict(Zs[(info.label == "RB100").values]).ravel().mean())
    return (float(np.sqrt(mean_squared_error(yc, p))),
            float(np.sqrt(mean_squared_error(pl.y, pl.p))), bias, p, c, yc, gc)

rows = []
for k in (1, 2, 4, 6, 8, 12):
    Zk, Mk = aggregate(Z, meta, keep_patch, k)
    for nc in (2, 3, 4, 5):
        u, pl, b, *_ = score(Zk, Mk, nc)
        rows.append(dict(k=k, n_comp=nc, units=int(Mk.label.isin(CAL_LABELS).sum()),
                         unit_rmsecv=u, plate_rmsecv=pl, day_bias=b))
S = pd.DataFrame(rows)
print(S.round(2).to_string(index=False))

fig, axs = plt.subplots(1, 3, figsize=(14, 3.4))
for nc in sorted(S.n_comp.unique()):
    s = S[S.n_comp == nc]
    axs[0].plot(s.k, s.plate_rmsecv, "o-", lw=1.3, label=f"{nc} LV")
    axs[1].plot(s.k, s.unit_rmsecv, "o-", lw=1.3)
    axs[2].plot(s.k, s.day_bias, "o-", lw=1.3)
axs[0].set_ylabel("plate RMSECV (pp)"); axs[0].set_title("plate error — the fair comparison")
axs[0].legend(fontsize=7)
axs[1].set_ylabel("unit RMSECV (pp)"); axs[1].set_title("per super-patch error")
axs[2].axhline(0, color="k", lw=.8); axs[2].set_ylabel("session bias (pp)")
axs[2].set_title("transfer to day 1")
for ax in axs: ax.set_xlabel("patches per super-patch")
plt.tight_layout(); plt.savefig(OUT / "02_selection.png", dpi=130, bbox_inches="tight"); plt.show()

# ---- automatic choice ----
# Two rules, both deliberate:
#   1. only configurations that TRANSFER (|session bias| <= 3 pp) are eligible, because every
#      sample you want to predict comes from the other session;
#   2. among those, take the SIMPLEST within 10 % of the best score, rather than the outright
#      minimum. The best raw score sits below the stir-to-stir reproducibility floor, which
#      means it is fitting measurement noise, not composition.
ok_bias = S[S.day_bias.abs() <= 3.0]
pool = ok_bias if len(ok_bias) else S
floor = pool.plate_rmsecv.min()
cands = pool[pool.plate_rmsecv <= floor * 1.10].sort_values(["n_comp", "k"])
pick = cands.iloc[0]

K_SEL  = int(pick.k)   if PATCHES_PER_UNIT == "auto" else int(PATCHES_PER_UNIT)
NC_SEL = int(pick.n_comp) if N_COMP == "auto" else int(N_COMP)

print(f"\nbest raw plate RMSECV in the grid : {S.plate_rmsecv.min():.2f} pp")
print(f"best among configurations that transfer: {floor:.2f} pp")
print(f"chosen (simplest within 10 % of that): k={K_SEL}, {NC_SEL} LV -> "
      f"{pick.plate_rmsecv:.2f} pp, session bias {pick.day_bias:+.2f}")

ZU, UM = aggregate(Z, meta, keep_patch, K_SEL)
N_COMP = NC_SEL
print(f"\n{len(ZU)} super-patches (median {int(np.median(UM.n_patches))} patches each)")
print(UM.groupby("label").agg(units=("sample_id", "size"), plates=("group", "nunique")).to_string())

## 4. The fitted model

In [ ]:
u_rmse, p_rmse, day_bias, pred_cv, calU, yU, gU = score(ZU, UM, N_COMP)
ZC = ZU[calU]
final = PLSRegression(n_components=N_COMP).fit(ZC, yU)

fold = pd.DataFrame([dict(level=lv, n_units=int((yU == lv).sum()),
                          bias=float(np.mean(pred_cv[yU == lv] - lv)),
                          rmse=float(np.sqrt(np.mean((pred_cv[yU == lv] - lv) ** 2))),
                          kind="extrapolated" if lv in (yU.min(), yU.max()) else "interpolated")
                     for lv in sorted(set(yU))])
print(fold.round(2).to_string(index=False))
interior = ~np.isin(yU, [yU.min(), yU.max()])
print(f"\nunit RMSECV  {u_rmse:.2f} pp   |  interior levels only {np.sqrt(np.mean((pred_cv[interior]-yU[interior])**2)):.2f}")
print(f"plate RMSECV {p_rmse:.2f} pp   |  R2 (CV) {r2_score(yU, pred_cv):.3f}")
print(f"session bias {day_bias:+.2f} pp  (day-2 model predicting day-1 RB100, truth 0)")

pm = pd.DataFrame(dict(y=yU, p=pred_cv, g=gU)).groupby(["g", "y"], as_index=False).agg(
    mean=("p", "mean"), sd=("p", "std"))
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].scatter(yU + RNG.normal(0, .8, len(yU)), pred_cv, s=9, alpha=.3, label="super-patches")
axs[0].errorbar(pm["y"], pm["mean"], yerr=pm["sd"], fmt="o", color="crimson", capsize=4,
                ms=6, lw=1.2, label="plate mean ± sd")
lims = [-8, 60]; axs[0].plot(lims, lims, "k--", lw=1); axs[0].set_xlim(lims); axs[0].set_ylim(lims)
axs[0].set_xlabel("true chicory %"); axs[0].set_ylabel("cross-validated prediction %")
axs[0].set_title(f"unit RMSECV {u_rmse:.2f} | plate {p_rmse:.2f} pp"); axs[0].legend(fontsize=7)

res = pred_cv - yU
cap = UM.capture.values[calU]
for c_ in sorted(set(cap)):
    s = cap == c_
    axs[1].scatter(yU[s] + (c_-2)*1.4, res[s], s=12, alpha=.6, label=f"stir {c_}")
axs[1].axhline(0, color="k", lw=.8); axs[1].set_xlabel("true chicory %")
axs[1].set_ylabel("residual (pp)"); axs[1].set_title("residuals by level and stir"); axs[1].legend(fontsize=7)

pc = UM.patch_col.values[calU]
axs[2].scatter(pc, res, s=10, alpha=.4)
zf = np.polyfit(pc, res, 1); xx = np.linspace(pc.min(), pc.max(), 40)
axs[2].plot(xx, np.poly1d(zf)(xx), "r-", lw=1.6); axs[2].axhline(0, color="k", lw=.8)
axs[2].set_xlabel("across-track position"); axs[2].set_ylabel("residual (pp)")
axs[2].set_title(f"vs position: {zf[0]*384:+.1f} pp across the swath")
plt.tight_layout(); plt.savefig(OUT / "03_fit.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
coef = final.coef_.ravel()
if coef.size != WL.size: coef = final.coef_.T.ravel()
Wt, Tt, Qt = final.x_weights_, final.x_scores_, final.y_loadings_.ravel()
ss = (Qt ** 2) * (Tt ** 2).sum(0)
vip = np.sqrt(WL.size * ((Wt ** 2 / (Wt ** 2).sum(0)) @ ss) / ss.sum())

fig, axs = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True)
axs[0].plot(WL, coef, lw=1.3); axs[0].axhline(0, color="k", lw=.6); axs[0].set_ylabel("PLS coefficient")
axs[1].plot(WL, vip, lw=1.3, color="darkorange"); axs[1].axhline(1, color="r", ls="--", lw=.8)
axs[1].set_ylabel("VIP"); axs[1].set_xlabel("wavelength (nm)")
for nm, t in [(1450, "O-H water"), (1730, "C-H lipid"), (1940, "O-H water"),
              (2100, "C-O carbohydrate"), (2310, "C-H combination")]:
    for a in axs: a.axvline(nm, color="0.7", lw=.8, ls=":")
    axs[0].text(nm, axs[0].get_ylim()[1], t, fontsize=6, rotation=90, va="top")
plt.tight_layout(); plt.savefig(OUT / "04_coefficients.png", dpi=130, bbox_inches="tight"); plt.show()
print("12 most influential wavelengths (nm):", np.sort(WL[np.argsort(-vip)[:12]]).astype(int))
print("Weight near 1730 nm is the C-H first overtone: coffee carries 10-15 % oil, chicory almost none.")

## 5. Validation

**Leave-one-concentration-out** is the number to report. **Leave-one-stir-out** is repeatability, not
validation — the same concentrations stay in training.

**The permutation test** is the one worth insisting on. With six levels and eighteen plates, a good
RMSECV can arise from the model separating a handful of objects and happening to order them. Shuffling
the concentrations across whole plates and refitting gives the null for exactly that.

In [ ]:
def cv_with(groups, nc=N_COMP):
    p = np.full(len(yU), np.nan)
    for tr, te in LeaveOneGroupOut().split(ZC, yU, groups=groups):
        p[te] = PLSRegression(n_components=nc).fit(ZC[tr], yU[tr]).predict(ZC[te]).ravel()
    return float(np.sqrt(mean_squared_error(yU, p)))

r_plate = cv_with(gU)
r_stir  = cv_with(UM.capture.values[calU])
print(f"leave-one-concentration-out {u_rmse:6.2f} pp   <- report this")
print(f"leave-one-plate-out         {r_plate:6.2f} pp")
print(f"leave-one-stir-out          {r_stir:6.2f} pp   (repeatability)")

py = pd.DataFrame(dict(g=gU, y=yU)).groupby("g")["y"].first()
null = []
for _ in range(N_PERM):
    mp = dict(zip(py.index, RNG.permutation(py.values)))
    yp = np.array([mp[g] for g in gU])
    p = np.full(len(yp), np.nan)
    for tr, te in LeaveOneGroupOut().split(ZC, yp, groups=yp):
        p[te] = PLSRegression(n_components=N_COMP).fit(ZC[tr], yp[tr]).predict(ZC[te]).ravel()
    null.append(np.sqrt(mean_squared_error(yp, p)))
null = np.array(null)
pval = (np.sum(null <= u_rmse) + 1) / (len(null) + 1)

fig, axs = plt.subplots(1, 2, figsize=(12, 3.4))
axs[0].bar(["level", "plate", "stir"], [u_rmse, r_plate, r_stir],
           color=["seagreen", "0.6", "0.6"])
axs[0].set_ylabel("RMSE (pp)"); axs[0].set_title("each split answers a different question")
axs[1].hist(null, bins=30, alpha=.8, color="0.6", label="shuffled labels")
axs[1].axvline(u_rmse, color="crimson", lw=2.2, label=f"real ({u_rmse:.2f})")
axs[1].set_xlabel("RMSECV (pp)"); axs[1].legend(fontsize=7)
axs[1].set_title(f"permutation test, n={N_PERM}, p = {pval:.3f}")
plt.tight_layout(); plt.savefig(OUT / "05_validation.png", dpi=130, bbox_inches="tight"); plt.show()
print(f"\np = {pval:.4f} -> " + ("not chance." if pval < 0.05 else "NOT significant."))

## 6. How stable are the unknowns?

Every modelling choice that seemed harmless moves the answers. Rather than pick one and quote it, the
predictions are recomputed across the plausible configurations — aggregation size, latent variables, and
whether the day-1 RB100 plate is included as the 0 % anchor.

**The spread across these is a real part of the uncertainty**, not an artefact. It exists because no plate
was measured on both days, so the session correction is inferred rather than measured.

In [ ]:
UNK = ["A", "B", "C"]
variants = []
for k in (4, 6, 8, 12):
    Zk, Mk = aggregate(Z, meta, keep_patch, k)
    ck = Mk.label.isin(CAL_LABELS).values
    for nc in (3, 4, 5):
        for anchor in ("with RB100", "day 2 only"):
            fitm = ck if anchor == "with RB100" else (ck & (Mk.day.values == 2))
            mdl = PLSRegression(n_components=nc).fit(Zk[fitm], Mk.chicory_pct.values[fitm])
            row = dict(k=k, n_comp=nc, anchor=anchor)
            for u in UNK:
                row[u] = float(mdl.predict(Zk[(Mk.label == u).values]).ravel().mean())
            row["RB100"] = float(mdl.predict(Zk[(Mk.label == "RB100").values]).ravel().mean())
            variants.append(row)
V = pd.DataFrame(variants)
print(V.round(1).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4))
for i, u in enumerate(UNK):
    ax.scatter([i]*len(V), V[u], s=26, alpha=.55)
    ax.plot([i-.25, i+.25], [V[u].mean()]*2, color="crimson", lw=2.5)
    ax.annotate(f"{V[u].mean():.1f} ± {V[u].std():.1f}", (i+.28, V[u].mean()), fontsize=9)
ax.axhspan(0, 50, color="seagreen", alpha=.10, label="calibration range")
ax.set_xticks(range(len(UNK)), UNK); ax.set_ylabel("predicted chicory %")
ax.set_title("every plausible configuration (18 of them)"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "06_stability.png", dpi=130, bbox_inches="tight"); plt.show()
CONFIG_SD = {u: float(V[u].std()) for u in UNK}
print("\nconfiguration spread (sd across variants):", {k: round(v, 1) for k, v in CONFIG_SD.items()})

## 7. Applicability domain

**Q residual** measures how far a spectrum lies off the model plane. Because day-1 spectra carry a
residual session signature, Q is compared against RB100 — the day-1 plate whose composition *is*
calibrated. The ratio removes the session part and leaves the composition part.

**This check catches composition, not species.** Simulated arabica–chicory mixtures give Q ratios of
0.3–1.0 — comfortably inside — while the predictions are wrong by 8–10 pp. Section 9 says more. Treat
this as "is the composition in range?", not "is this the right coffee?".

In [ ]:
mu = ZC.mean(0); Rot = final.x_rotations_; P = final.x_loadings_
def QT(S):
    S = np.atleast_2d(S); T = (S - mu) @ Rot
    return np.sum(((S - mu) - T @ P.T) ** 2, 1), T
Qc, Tc = QT(ZC)
Si = np.linalg.pinv(np.atleast_2d(np.cov(Tc, rowvar=False)) + np.eye(Tc.shape[1]) * 1e-12)
n_, A_ = Tc.shape
t2lim = A_ * (n_ - 1) / (n_ - A_) * f_dist.ppf(.95, A_, n_ - A_)
q_ref = float(np.median(QT(ZU[(UM.label == "RB100").values])[0]))

dom = []
for lab in ["RB100"] + CAL_LABELS[1:] + ["A", "B", "C", "CH100"]:
    s = (UM.label == lab).values
    if s.sum() == 0: continue
    q, t = QT(ZU[s])
    dom.append(dict(label=lab, n=int(s.sum()), Q_rel=float(np.median(q) / q_ref),
                    T2_out=float(np.mean(np.einsum("ij,jk,ik->i", t, Si, t) > t2lim))))
D = pd.DataFrame(dom)
print(D.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 3.6))
cols_ = ["seagreen" if v < 2 else "crimson" for v in D.Q_rel]
ax.barh(D.label, D.Q_rel, color=cols_)
ax.axvline(1, color="k", lw=1.2); ax.axvline(2, color="r", ls="--", lw=1)
ax.set_xlabel("median Q ÷ RB100's Q"); ax.set_title("composition inside the calibrated space?")
plt.tight_layout(); plt.savefig(OUT / "07_domain.png", dpi=130, bbox_inches="tight"); plt.show()

## 8. Results

In [ ]:
def report(lab):
    s = (UM.label == lab).values
    p = final.predict(ZU[s]).ravel()
    per_cap = pd.Series(p).groupby(UM.capture.values[s]).mean()
    q_rel = float(np.median(QT(ZU[s])[0]) / q_ref)
    u_stir = float(per_cap.std(ddof=1)) if len(per_cap) > 1 else 0.0
    u_cfg = CONFIG_SD.get(lab, 0.0)
    u = float(np.sqrt(u_rmse ** 2 + u_stir ** 2 + u_cfg ** 2))
    inside = q_rel < 2 and -5 <= p.mean() <= 55
    return dict(sample=lab, chicory_pct=round(float(p.mean()), 1),
                robusta_pct=round(100 - float(p.mean()), 1) if inside else np.nan,
                uncertainty=round(u, 1), model_err=round(u_rmse, 1),
                stir=round(u_stir, 1), config=round(u_cfg, 1), Q_rel=round(q_rel, 2),
                verdict="quantifiable" if inside else "beyond the 0-50 % range — detect, don't quantify")

RES = pd.DataFrame([report(l) for l in UNK + ["RB100", "CH100"]])
print(RES.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4))
sub = RES[RES["sample"].isin(UNK)]
ax.errorbar(range(len(sub)), sub.chicory_pct, yerr=sub.uncertainty, fmt="o", capsize=6, ms=10)
for i, r_ in enumerate(sub.itertuples()):
    ok_ = "quant" in r_.verdict
    ax.annotate(f"{r_.chicory_pct:.1f} ± {r_.uncertainty:.1f}" + ("" if ok_ else "  (beyond range)"),
                (i, r_.chicory_pct), xytext=(14, 0), textcoords="offset points", fontsize=9,
                color="k" if ok_ else "crimson")
ax.axhspan(0, 50, color="seagreen", alpha=.10, label="calibration range")
ax.set_xticks(range(len(sub)), sub["sample"]); ax.set_ylabel("chicory %")
ax.set_ylim(0, 100); ax.set_xlim(-.5, len(sub) - .2)
ax.set_title("robusta–chicory model: final estimates"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "08_results.png", dpi=130, bbox_inches="tight"); plt.show()

summary = dict(model="PLS, absorbance + offset removal",
               n_latent=int(N_COMP), patches_per_unit=int(K_SEL),
               trim_pct=int(TRIM_WORST_PCT), wl_range=[float(WL[0]), float(WL[-1])],
               levels=sorted(map(float, set(yU))), n_units=int(calU.sum()),
               n_plates=int(len(set(gU))),
               rmsecv_unit=round(u_rmse, 2), rmsecv_plate=round(p_rmse, 2),
               rmsecv_stir=round(r_stir, 2), r2_cv=round(float(r2_score(yU, pred_cv)), 4),
               session_bias=round(day_bias, 2), permutation_p=round(float(pval), 4),
               validity_range=[0, 50],
               predictions={r["sample"]: {k: (None if pd.isna(v) else v)
                                          for k, v in r.items() if k != "sample"}
                            for _, r in RES.iterrows()})
(OUT / "summary.json").write_text(json.dumps(summary, indent=2))
RES.to_csv(OUT / "predictions.csv", index=False)
print(json.dumps(summary, indent=2))

## 9. Limits

**Range.** Calibration covers 0–50 % chicory across six levels. There is nothing between 50 and 100 %, so
anything predicted above ~50 % is extrapolating across a gap half the width of the scale. Detection is
sound there; quantification is not.

**Species.** Arabica was dropped from this model, and that leaves a real hole. Simulated arabica–chicory
mixtures return Q ratios of 0.3–1.0 — inside the domain — while predictions run 8–10 pp low. The model
does not measure chicory; it measures displacement from robusta along the one direction it was shown. Any
other adulterant — corn, barley, husks, caramel — would be projected onto the same axis and reported as
chicory. A species gate before the regression would close this, and the data for one already exists.

**Sessions.** Your mixtures are day 2 and RB100 and all the unknowns are day 1. No plate was measured on
both, so the correction is inferred from the gap's multiplicative structure rather than measured against a
transfer standard. It verifies well — reflectance scaled ×0.85 to ×1.15 changes the prediction by zero —
but it rests on an assumption.

**Effective sample size.** Six concentrations, eighteen plates, **one preparation per level**. The super-
patch count is not your sample size. The model has never seen a concentration prepared twice, so it cannot
fully separate composition from that particular grind and packing.

**Uniformity.** One roast, one grind, one moisture state. Water dominates the raw spectra at 1450 and
1940 nm, and nothing in the calibration varies it.

### Worth doing next, in order

1. One plate scanned in both sessions — turns the session correction from inferred to measured.
2. Levels at 60, 70, 80, 90 % chicory — makes samples like B and C quantifiable and lets CH100 in.
3. Two independent weighings per level — the single real constraint on the model's degrees of freedom.
4. A robusta/arabica gate before the regression, to close the silent-failure path.